# Assignment 1: Network Structure Analysis
**CIEQ6232 — Academic year 2025/26 — Q4**

Set the `CITY`, `MODE`, and `DB_PATH` in the **Config** cell below, then run all cells.

In [ ]:
# ============================================================
# CONFIG — edit these three lines before running
# ============================================================
CITY = "TODO"          # e.g. "chicago", "berlin", "paris"
MODE = "TODO"          # one of: "Tram", "Subway", "Rail", "Bus"
DB_PATH = f"data/{CITY}.sqlite"   # path to the GTFS SQLite file
# ============================================================

In [ ]:
import sqlite3
import pickle
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from scipy import stats
import folium

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

# GTFS route_type codes
MODE_TO_ROUTE_TYPE = {
    'Tram':   0,
    'Subway': 1,
    'Rail':   2,
    'Bus':    3,
}
ROUTE_TYPE = MODE_TO_ROUTE_TYPE[MODE]
print(f"City: {CITY}  |  Mode: {MODE}  |  route_type: {ROUTE_TYPE}")

## Helper functions

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    """Great-circle distance in metres."""
    R = 6_371_000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def parse_gtfs_time(t):
    """Parse GTFS time string HH:MM:SS (hours may exceed 23) → seconds."""
    h, m, s = map(int, str(t).split(':'))
    return h * 3600 + m * 60 + s


def print_indicator(name, values_dict):
    """Pretty-print a global indicator."""
    for label, val in values_dict.items():
        print(f"  {name} [{label}]: {val}")


def local_stats(centrality_dict, graph, name, top=3):
    """Return a dict with mean, std, min, max, and top-k nodes."""
    vals = np.array(list(centrality_dict.values()))
    top_nodes = sorted(centrality_dict, key=centrality_dict.get, reverse=True)[:top]
    top_list = [(graph.nodes[n].get('stop_name', n), round(centrality_dict[n], 6)) for n in top_nodes]
    result = {
        'mean': vals.mean(),
        'std':  vals.std(),
        'min':  vals.min(),
        'max':  vals.max(),
        f'top_{top}': top_list,
    }
    print(f"\n{name}")
    print(f"  mean={result['mean']:.6f}  std={result['std']:.6f}  "
          f"min={result['min']:.6f}  max={result['max']:.6f}")
    print(f"  top {top}: {top_list}")
    return result

## 1 — Load GTFS data

In [ ]:
con = sqlite3.connect(DB_PATH)

# Routes of the selected mode
routes = pd.read_sql(
    f"SELECT route_id, route_short_name, route_long_name, route_type "
    f"FROM routes WHERE route_type = {ROUTE_TYPE}",
    con
)
print(f"Routes of type '{MODE}': {len(routes)}")

# All trips for these routes
route_ids_sql = "','".join(routes['route_id'].tolist())
trips = pd.read_sql(
    f"SELECT trip_id, route_id FROM trips WHERE route_id IN ('{route_ids_sql}')",
    con
)
print(f"Trips: {len(trips)}")

# Stop times for those trips
trip_ids_sql = "','".join(trips['trip_id'].tolist())
stop_times = pd.read_sql(
    f"SELECT trip_id, arrival_time, departure_time, stop_id, stop_sequence "
    f"FROM stop_times WHERE trip_id IN ('{trip_ids_sql}') "
    f"ORDER BY trip_id, stop_sequence",
    con
)
stop_times['dep_sec'] = stop_times['departure_time'].apply(parse_gtfs_time)
stop_times['arr_sec'] = stop_times['arrival_time'].apply(parse_gtfs_time)
print(f"Stop-time records: {len(stop_times)}")

# Stop metadata
stop_ids_sql = "','".join(stop_times['stop_id'].unique().tolist())
stops = pd.read_sql(
    f"SELECT stop_id, stop_name, stop_lat, stop_lon FROM stops "
    f"WHERE stop_id IN ('{stop_ids_sql}')",
    con
)
stops = stops.set_index('stop_id')
print(f"Unique stops: {len(stops)}")

# Merge trip → route mapping into stop_times
stop_times = stop_times.merge(trips[['trip_id', 'route_id']], on='trip_id', how='left')
con.close()

## 2 — Build L-space graph

**Nodes:** stops (with lat/lon).  
**Edges:** consecutive stops along a trip.  
**Edge weights:** average in-vehicle time (seconds) and average link length (metres).

In [ ]:
G_L = nx.Graph()

# Add nodes
for stop_id, row in stops.iterrows():
    G_L.add_node(stop_id,
                 stop_name=row['stop_name'],
                 lat=row['stop_lat'],
                 lon=row['stop_lon'])

# Collect edge travel times across all trips
edge_times = {}   # (u, v) → list of travel times in seconds

for trip_id, group in stop_times.sort_values('stop_sequence').groupby('trip_id'):
    seq = group.reset_index(drop=True)
    for i in range(len(seq) - 1):
        u = seq.loc[i, 'stop_id']
        v = seq.loc[i+1, 'stop_id']
        t_travel = seq.loc[i+1, 'arr_sec'] - seq.loc[i, 'dep_sec']
        if t_travel <= 0:
            continue  # skip invalid records
        key = tuple(sorted([u, v]))
        edge_times.setdefault(key, []).append(t_travel)

# Add edges with average travel time and haversine distance
for (u, v), times in edge_times.items():
    if u not in stops.index or v not in stops.index:
        continue
    avg_time = np.mean(times)   # seconds
    dist_m   = haversine(
        stops.loc[u, 'stop_lat'], stops.loc[u, 'stop_lon'],
        stops.loc[v, 'stop_lat'], stops.loc[v, 'stop_lon']
    )
    G_L.add_edge(u, v, travel_time=avg_time, length=dist_m, weight=avg_time)

# Keep largest connected component
largest_cc = max(nx.connected_components(G_L), key=len)
G_L = G_L.subgraph(largest_cc).copy()

print(f"L-space  |  nodes: {G_L.number_of_nodes()}  |  edges: {G_L.number_of_edges()}")

## 3 — Build P-space graph

**Nodes:** same stops as L-space.  
**Edges:** all pairs of stops served by the same route (fully-connected per route).  
**Edge weight:** average waiting time = headway / 2 (minutes).

In [ ]:
# Compute per-route headway → average waiting time
# Headway = average time between consecutive departures at the first stop of each route
route_waiting = {}   # route_id → avg waiting time (minutes)

for route_id, r_group in stop_times.groupby('route_id'):
    # departures at the most-served stop
    first_stop = r_group.groupby('stop_id')['trip_id'].count().idxmax()
    deps = r_group[r_group['stop_id'] == first_stop]['dep_sec'].sort_values().values
    if len(deps) > 1:
        headways = np.diff(deps)
        headways = headways[headways > 0]  # remove negative wraps
        if len(headways) > 0:
            avg_headway_min = np.median(headways) / 60
            route_waiting[route_id] = avg_headway_min / 2
        else:
            route_waiting[route_id] = np.nan
    else:
        route_waiting[route_id] = np.nan

global_avg_wait = np.nanmean(list(route_waiting.values()))
print(f"Global average waiting time: {global_avg_wait:.2f} min")

# Build P-space
G_P = nx.Graph()

# Nodes: only those present in L-space (largest CC)
lspace_nodes = set(G_L.nodes())
for n in lspace_nodes:
    G_P.add_node(n, **G_L.nodes[n])

pspace_edge_waits = {}  # (u,v) → list of waiting times

for route_id, r_group in stop_times.groupby('route_id'):
    route_stops = [s for s in r_group['stop_id'].unique() if s in lspace_nodes]
    wait = route_waiting.get(route_id, global_avg_wait)
    if np.isnan(wait):
        wait = global_avg_wait
    # All pairs
    for i in range(len(route_stops)):
        for j in range(i+1, len(route_stops)):
            key = tuple(sorted([route_stops[i], route_stops[j]]))
            pspace_edge_waits.setdefault(key, []).append(wait)

for (u, v), waits in pspace_edge_waits.items():
    avg_wait = np.mean(waits)
    G_P.add_edge(u, v, waiting_time=avg_wait, weight=avg_wait)

# Keep only nodes that are in L-space largest CC
G_P = G_P.subgraph(lspace_nodes).copy()

print(f"P-space  |  nodes: {G_P.number_of_nodes()}  |  edges: {G_P.number_of_edges()}")

## 4 — Save / reload Network.pkl

In [ ]:
Path('output').mkdir(exist_ok=True)

network = {'L_space': G_L, 'P_space': G_P, 'city': CITY, 'mode': MODE}
with open('output/Network.pkl', 'wb') as f:
    pickle.dump(network, f)
print("Saved output/Network.pkl")

# To reload:
# with open('output/Network.pkl', 'rb') as f:
#     network = pickle.load(f)
# G_L = network['L_space'];  G_P = network['P_space']

## 5 — Global network indicators

In [ ]:
def graph_diameter(G, weight=None):
    """Diameter of a graph (max shortest path). Uses weight attribute if given."""
    return nx.diameter(G, weight=weight)

def avg_shortest_path(G, weight=None):
    """Average shortest path length."""
    return nx.average_shortest_path_length(G, weight=weight)

# --- node / edge counts ---
N_L, E_L = G_L.number_of_nodes(), G_L.number_of_edges()
N_P, E_P = G_P.number_of_nodes(), G_P.number_of_edges()

print("=" * 55)
print("GLOBAL INDICATORS")
print("=" * 55)
print(f"Nodes — L-space: {N_L}   P-space: {N_P}")
print(f"Edges — L-space: {E_L}   P-space: {E_P}")

# --- diameter ---
# NOTE: for large networks these can be slow; subsample if needed
print("\nComputing diameter (this may take a while for large networks)...")
diam_L_uw = graph_diameter(G_L, weight=None)
diam_L_w  = graph_diameter(G_L, weight='travel_time')
diam_P_uw = graph_diameter(G_P, weight=None)
diam_P_w  = graph_diameter(G_P, weight='waiting_time')

print(f"Diameter — L unweighted: {diam_L_uw}   L weighted: {diam_L_w:.1f} s")
print(f"Diameter — P unweighted: {diam_P_uw}   P weighted: {diam_P_w:.2f} min")

# --- average shortest path ---
print("\nComputing ASP...")
asp_L_uw = avg_shortest_path(G_L, weight=None)
asp_L_w  = avg_shortest_path(G_L, weight='travel_time')
asp_P_uw = avg_shortest_path(G_P, weight=None)
asp_P_w  = avg_shortest_path(G_P, weight='waiting_time')

print(f"ASP — L unweighted: {asp_L_uw:.4f}   L weighted: {asp_L_w:.1f} s")
print(f"ASP — P unweighted: {asp_P_uw:.4f}   P weighted: {asp_P_w:.4f} min")

# --- connectivity (gamma index) — L-space only ---
# For planar graph: gamma = E / (3*(N-2))
gamma = E_L / (3 * (N_L - 2)) if N_L > 2 else float('nan')

# --- meshedness (alpha index) — L-space only ---
# alpha = (E - N + 1) / (2*N - 5)
alpha = (E_L - N_L + 1) / (2 * N_L - 5) if N_L > 2 else float('nan')

print(f"\nConnectivity (gamma / γ) — L-space: {gamma:.4f}")
print(f"Meshedness   (alpha / α) — L-space: {alpha:.4f}")

# Collect into a summary DataFrame
global_summary = pd.DataFrame({
    'Indicator': [
        'N (L-space)', 'N (P-space)',
        'E (L-space)', 'E (P-space)',
        'Diameter L unweighted', 'Diameter L weighted (s)',
        'Diameter P unweighted', 'Diameter P weighted (min)',
        'ASP L unweighted', 'ASP L weighted (s)',
        'ASP P unweighted', 'ASP P weighted (min)',
        'Gamma (connectivity)', 'Alpha (meshedness)',
    ],
    'Value': [
        N_L, N_P, E_L, E_P,
        diam_L_uw, diam_L_w, diam_P_uw, diam_P_w,
        asp_L_uw, asp_L_w, asp_P_uw, asp_P_w,
        gamma, alpha,
    ]
})
global_summary.to_csv('output/global_indicators.csv', index=False)
print("\nSaved output/global_indicators.csv")

## 6 — Local indicators: L-space

In [ ]:
print("=" * 55)
print("LOCAL INDICATORS — L-SPACE")
print("=" * 55)

# Degree centrality (same for weighted/unweighted in undirected)
deg_L = nx.degree_centrality(G_L)
stats_deg_L = local_stats(deg_L, G_L, "Degree centrality — L-space")

# Closeness centrality
close_L_uw = nx.closeness_centrality(G_L, distance=None)
close_L_w  = nx.closeness_centrality(G_L, distance='travel_time')
stats_close_L_uw = local_stats(close_L_uw, G_L, "Closeness centrality — L-space unweighted")
stats_close_L_w  = local_stats(close_L_w,  G_L, "Closeness centrality — L-space weighted")

# Betweenness centrality (relative = normalised by default in NetworkX)
print("\nComputing betweenness centrality (may be slow)...")
between_L_uw = nx.betweenness_centrality(G_L, weight=None,          normalized=True)
between_L_w  = nx.betweenness_centrality(G_L, weight='travel_time', normalized=True)
stats_between_L_uw = local_stats(between_L_uw, G_L, "Betweenness centrality — L-space unweighted")
stats_between_L_w  = local_stats(between_L_w,  G_L, "Betweenness centrality — L-space weighted")

## 7 — Local indicators: P-space

In [ ]:
print("=" * 55)
print("LOCAL INDICATORS — P-SPACE")
print("=" * 55)

deg_P = nx.degree_centrality(G_P)
stats_deg_P = local_stats(deg_P, G_P, "Degree centrality — P-space")

close_P_uw = nx.closeness_centrality(G_P, distance=None)
close_P_w  = nx.closeness_centrality(G_P, distance='waiting_time')
stats_close_P_uw = local_stats(close_P_uw, G_P, "Closeness centrality — P-space unweighted")
stats_close_P_w  = local_stats(close_P_w,  G_P, "Closeness centrality — P-space weighted")

print("\nComputing betweenness centrality (may be slow)...")
between_P_uw = nx.betweenness_centrality(G_P, weight=None,           normalized=True)
between_P_w  = nx.betweenness_centrality(G_P, weight='waiting_time', normalized=True)
stats_between_P_uw = local_stats(between_P_uw, G_P, "Betweenness centrality — P-space unweighted")
stats_between_P_w  = local_stats(between_P_w,  G_P, "Betweenness centrality — P-space weighted")

## 8 — Histograms of all centrality indicators

In [ ]:
centrality_sets = [
    (deg_L,         'Degree — L-space'),
    (close_L_uw,    'Closeness L-space (unweighted)'),
    (close_L_w,     'Closeness L-space (weighted)'),
    (between_L_uw,  'Betweenness L-space (unweighted)'),
    (between_L_w,   'Betweenness L-space (weighted)'),
    (deg_P,         'Degree — P-space'),
    (close_P_uw,    'Closeness P-space (unweighted)'),
    (close_P_w,     'Closeness P-space (weighted)'),
    (between_P_uw,  'Betweenness P-space (unweighted)'),
    (between_P_w,   'Betweenness P-space (weighted)'),
]

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for ax, (c_dict, title) in zip(axes.flat, centrality_sets):
    vals = list(c_dict.values())
    ax.hist(vals, bins=30, color='steelblue', edgecolor='white', linewidth=0.4)
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('Value', fontsize=7)
    ax.set_ylabel('Count', fontsize=7)
    ax.tick_params(labelsize=6)

plt.suptitle(f"{CITY.title()} — Centrality Histograms", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('output/histograms.png', bbox_inches='tight')
plt.show()
print("Saved output/histograms.png")

## 9 — Map visualisations (weighted L-space centralities)

In [ ]:
def make_centrality_map(G, centrality_dict, title, filename, cmap='RdYlGn'):
    """Create a folium map with nodes coloured by centrality value."""
    vals = np.array(list(centrality_dict.values()))
    norm = mcolors.Normalize(vmin=vals.min(), vmax=vals.max())
    colormap = cm.get_cmap(cmap)

    lats = [G.nodes[n]['lat'] for n in G.nodes()]
    lons = [G.nodes[n]['lon'] for n in G.nodes()]
    m = folium.Map(location=[np.mean(lats), np.mean(lons)], zoom_start=12,
                   tiles='CartoDB positron')

    # Draw edges
    for u, v in G.edges():
        lat_u, lon_u = G.nodes[u]['lat'], G.nodes[u]['lon']
        lat_v, lon_v = G.nodes[v]['lat'], G.nodes[v]['lon']
        folium.PolyLine([[lat_u, lon_u], [lat_v, lon_v]],
                        color='#888888', weight=1, opacity=0.5).add_to(m)

    # Draw nodes
    for node in G.nodes():
        c_val = centrality_dict.get(node, 0)
        rgba  = colormap(norm(c_val))
        color = mcolors.to_hex(rgba)
        radius = 4 + 14 * norm(c_val)   # size scales with centrality
        folium.CircleMarker(
            location=[G.nodes[node]['lat'], G.nodes[node]['lon']],
            radius=radius,
            color=color, fill=True, fill_color=color, fill_opacity=0.85,
            tooltip=f"{G.nodes[node].get('stop_name', node)}: {c_val:.4f}"
        ).add_to(m)

    m.save(f'output/{filename}.html')
    print(f"Saved output/{filename}.html")


make_centrality_map(G_L, deg_L,        f"{CITY} — Degree centrality (L-space)",      'map_degree_L')
make_centrality_map(G_L, close_L_w,    f"{CITY} — Closeness centrality (L-space, w)", 'map_closeness_L_w')
make_centrality_map(G_L, between_L_w,  f"{CITY} — Betweenness centrality (L-space, w)",'map_betweenness_L_w')

## 10 — Save centrality results to CSV (for Brightspace upload)

In [ ]:
# Build a combined DataFrame indexed by stop_id
df_centrality = pd.DataFrame(index=list(G_L.nodes()))
df_centrality['stop_name']      = [G_L.nodes[n].get('stop_name', n) for n in df_centrality.index]
df_centrality['deg_L']          = pd.Series(deg_L)
df_centrality['close_L_uw']     = pd.Series(close_L_uw)
df_centrality['close_L_w']      = pd.Series(close_L_w)
df_centrality['between_L_uw']   = pd.Series(between_L_uw)
df_centrality['between_L_w']    = pd.Series(between_L_w)
df_centrality['deg_P']          = pd.Series(deg_P)
df_centrality['close_P_uw']     = pd.Series(close_P_uw)
df_centrality['close_P_w']      = pd.Series(close_P_w)
df_centrality['between_P_uw']   = pd.Series(between_P_uw)
df_centrality['between_P_w']    = pd.Series(between_P_w)

df_centrality.to_csv('output/centrality_indicators.csv')
print("Saved output/centrality_indicators.csv")
df_centrality.describe()

## 11 — Pearson correlations

### Part B — Student #1: within a single representation
Correlations between all pairs of local indicators *within* L-space (or P-space).

### Part B — Student #2: across representations
Correlations between indicators *across* L-space and P-space.

In [ ]:
# --- Student #1: within L-space ---
lspace_cols = ['deg_L', 'close_L_uw', 'close_L_w', 'between_L_uw', 'between_L_w']
corr_L = df_centrality[lspace_cols].corr(method='pearson')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(corr_L, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[0], linewidths=0.5)
axes[0].set_title('Pearson correlations — within L-space', fontsize=11)

# --- Student #2: across spaces ---
# Example pairs: degree_L vs degree_P, closeness_L_uw vs closeness_P_uw, etc.
cross_cols = ['deg_L', 'close_L_w', 'between_L_w', 'deg_P', 'close_P_w', 'between_P_w']
corr_cross = df_centrality[cross_cols].corr(method='pearson')

sns.heatmap(corr_cross, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            ax=axes[1], linewidths=0.5)
axes[1].set_title('Pearson correlations — across L-space & P-space', fontsize=11)

plt.tight_layout()
plt.savefig('output/pearson_correlations.png', bbox_inches='tight')
plt.show()
print("Saved output/pearson_correlations.png")

## 12 — Efficiency & Robustness indicators (Step 5)

### Efficiency (Student #1)
$$E = \\bar{C}_{uw} \\cdot (1 - \\bar{b}_{uw})$$
- $\\bar{C}_{uw}$ — average unweighted L-space closeness centrality (high = stations easily
  reachable from anywhere → compact, efficiently traversed network).
- $(1 - \\bar{b}_{uw})$ — penalises routing concentration: high average betweenness means
  a few nodes absorb most shortest-path traffic, reducing *practical* efficiency.
- Both factors ∈ [0, 1] and normalised by NetworkX for network size, allowing
  fair comparison across networks of different sizes.

### Robustness (Student #2)
$$R = \\bar{C}_{uw} \\cdot \\frac{\\bar{b}_{uw}}{b_{uw,\\max}}$$
- $\\bar{C}_{uw}$ — a well-connected network can better absorb node failures
  (stations remain reachable via alternative paths).
- $\\bar{b}_{uw} / b_{uw,\\max}$ — *betweenness uniformity ratio*: close to 1 when routing
  load is evenly distributed (no single critical bottleneck); close to 0 when one node
  dominates all shortest paths and its removal would cripple the network.
- Both terms ∈ (0, 1] → product ∈ (0, 1] and size-normalised.

**Shared ingredient:** $\\bar{C}_{uw}$ appears in both formulas (permitted by the assignment).

In [ ]:
# ---------------------------------------------------------------
# EFFICIENCY (Student #1)
# E = avg_closeness_uw × (1 − avg_betweenness_uw)
#
# ROBUSTNESS (Student #2)
# R = avg_closeness_uw × (avg_betweenness_uw / max_betweenness_uw)
# ---------------------------------------------------------------

close_uw_vals = list(close_L_uw.values())
betw_uw_vals  = list(between_L_uw.values())

c_avg = np.mean(close_uw_vals)
b_avg = np.mean(betw_uw_vals)
b_max = max(betw_uw_vals)

efficiency = c_avg * (1 - b_avg)
robustness = c_avg * (b_avg / b_max) if b_max > 0 else float('nan')

print(f"Efficiency = {efficiency:.6f}")
print(f"  avg_close_uw = {c_avg:.4f}")
print(f"  1 - avg_betw_uw = {1 - b_avg:.4f}")
print()
print(f"Robustness = {robustness:.6f}")
print(f"  avg_betw_uw = {b_avg:.4f}")
print(f"  max_betw_uw = {b_max:.4f}")
print(f"  betw uniformity ratio = {b_avg/b_max:.4f}")

In [ ]:
# ----------------------------------------------------------------
# Scatter plot: Efficiency vs Robustness — all submitted networks
# Loads data/shared_indicators.csv (generated by data/make_shared_csv.py)
# and appends the own network computed in cell 29.
# ----------------------------------------------------------------
import os

shared_path = 'data/shared_indicators.csv'
if not os.path.exists(shared_path):
    print(f"ERROR: {shared_path} not found — run data/make_shared_csv.py first.")
else:
    df_shared = pd.read_csv(shared_path)

    # Compute composite indicators for each shared network
    df_shared['efficiency'] = (
        df_shared['close_uw_avg'] * (1 - df_shared['betw_uw_avg'])
    )
    df_shared['robustness'] = (
        df_shared['close_uw_avg']
        * df_shared['betw_uw_avg']
        / df_shared['betw_uw_max']
    )

    df_plot = df_shared.dropna(subset=['efficiency', 'robustness']).copy()
    df_plot['is_own'] = False

    # Add own network (computed in cell 29)
    own_label = CITY.title() if CITY != 'TODO' else 'My Network'
    df_own = pd.DataFrame([{
        'network': own_label,
        'mode': MODE,
        'efficiency': efficiency,
        'robustness': robustness,
        'is_own': True,
    }])
    df_plot = pd.concat([df_plot, df_own], ignore_index=True)

    # ── plot ──────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(13, 8))

    mask_other = ~df_plot['is_own']
    ax.scatter(
        df_plot.loc[mask_other, 'efficiency'],
        df_plot.loc[mask_other, 'robustness'],
        s=70, color='steelblue', alpha=0.8, zorder=3, label='Other networks'
    )
    ax.scatter(
        df_plot.loc[df_plot['is_own'], 'efficiency'],
        df_plot.loc[df_plot['is_own'], 'robustness'],
        s=200, color='crimson', zorder=5, marker='*', label=own_label
    )

    for _, row in df_plot.iterrows():
        ax.annotate(
            row['network'],
            (row['efficiency'], row['robustness']),
            textcoords='offset points', xytext=(5, 3), fontsize=7
        )

    ax.set_xlabel(
        r'Efficiency  $=\,\bar{C}_{uw}\cdot(1-\bar{b}_{uw})$',
        fontsize=11
    )
    ax.set_ylabel(
        r'Robustness  $=\,\bar{C}_{uw}\cdot\bar{b}_{uw}\,/\,b_{uw,\max}$',
        fontsize=11
    )
    ax.set_title('Network Efficiency vs Robustness — all submitted networks', fontsize=13)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle='--', alpha=0.4)

    Path('output').mkdir(exist_ok=True)
    plt.tight_layout()
    plt.savefig('output/efficiency_robustness_scatter.png', bbox_inches='tight')
    plt.show()
    print(f"Saved output/efficiency_robustness_scatter.png")

    # Summary table
    print()
    summary = (
        df_plot[['network', 'efficiency', 'robustness']]
        .sort_values('efficiency', ascending=False)
        .reset_index(drop=True)
    )
    print(summary.to_string(index=False))